# Lab 12 — Ridge/Lasso Regression and Polynomial Features

In this lab you will regularize a linear model with Ridge and Lasso, use cross-validation to select the regularization strength, and see what happens when you combine polynomial feature expansion with regularization to control the resulting complexity.

The pipeline pattern is the same one you have used in every regression lab so far: split, fit, predict, evaluate. Two new pieces are added here: feature scaling, which regularization requires, and a penalty term that controls how much the model can rely on any one feature.

**Concepts covered:** Ridge (L2) and Lasso (L1) regularization, coefficient shrinkage vs. automatic feature selection, cross-validated alpha selection, feature scaling, polynomial feature expansion.

**Reference working sessions:**
- `working-sessions/supervised/04_ridge_lasso_regression.ipynb`
- `working-sessions/feature_engineering/05_polynomial_features.ipynb`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import Ridge, Lasso, RidgeCV, LassoCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import r2_score

from tkh_utils import (
    PALETTE, FONT, base_layout,
    check_answer, make_answer_key, make_grading_summary,
    load_california_housing,
)

_ak = make_answer_key({
    'q1': 'B',
    'q2': 'A',
    'q3': 'B',
    'q4': 'B',
})

---
## Section A — Multiple Choice

Fill in each answer variable with the letter of the best answer (A, B, C, or D).

In [ ]:
# Q1 — What is the key difference between how Ridge (L2) and Lasso (L1)
# regularization affect a model's coefficients?
#
#   A) Ridge sets some coefficients exactly to zero; Lasso only shrinks
#      them slightly
#   B) Ridge shrinks all coefficients toward zero proportionally without
#      eliminating any; Lasso can drive some coefficients to exactly zero,
#      effectively removing those features
#   C) Ridge and Lasso produce identical coefficient paths, just computed
#      with different algorithms
#   D) Ridge only affects the intercept, while Lasso only affects the
#      feature coefficients

q1_answer = "___"  # Replace with A, B, C, or D

assert q1_answer != "___", "Don't forget to fill in your answer!"
assert check_answer(q1_answer, _ak['q1']), \
    "Not quite — revisit working-sessions/supervised/04_ridge_lasso_regression.ipynb. " \
    "Look at the Coefficient Shrinkage Explorer and compare what happens to each coefficient as alpha grows."
print("✓ Question 1 correct!")

In [ ]:
# Q2 — Why does Lasso perform automatic feature selection while Ridge does not?
#
#   A) Lasso's L1 penalty has sharp corners in its constraint region, so the
#      optimal solution often lands exactly on an axis where a coefficient
#      is zero; Ridge's smooth L2 constraint region rarely produces exact
#      zeros
#   B) Lasso uses a much larger alpha by default than Ridge
#   C) Lasso only works on categorical features, so numeric coefficients are
#      automatically dropped
#   D) Ridge cannot be used with more than 10 features, so Lasso is preferred
#      for feature selection

q2_answer = "___"  # Replace with A, B, C, or D

assert q2_answer != "___", "Don't forget to fill in your answer!"
assert check_answer(q2_answer, _ak['q2']), \
    "Not quite — revisit working-sessions/supervised/04_ridge_lasso_regression.ipynb. " \
    "Look at the geometric explanation comparing the L1 diamond to the L2 sphere."
print("✓ Question 2 correct!")

In [ ]:
# Q3 — Why do polynomial feature counts grow so quickly with degree?
#
#   A) sklearn adds a fixed number of extra features regardless of degree
#   B) PolynomialFeatures adds every combination of input features up to
#      the chosen degree, including interaction terms, so the count grows
#      combinatorially with the number of original features and the degree
#   C) Higher-degree polynomials require duplicating the dataset once per
#      degree
#   D) Scaling doubles the number of features at each degree

q3_answer = "___"  # Replace with A, B, C, or D

assert q3_answer != "___", "Don't forget to fill in your answer!"
assert check_answer(q3_answer, _ak['q3']), \
    "Not quite — revisit working-sessions/feature_engineering/05_polynomial_features.ipynb. " \
    "Look at the feature-count table — 10 inputs become 65 features at degree 2."
print("✓ Question 3 correct!")

In [ ]:
# Q4 — Why do Ridge and Lasso require feature scaling before fitting?
#
#   A) Scaling makes the model train faster but has no effect on the result
#   B) Without scaling, features with larger numeric ranges get penalized
#      less relative to their actual importance, since the penalty term
#      treats every coefficient on the same absolute scale
#   C) Regularized models cannot accept negative values, so scaling is
#      required to make all features positive
#   D) Scaling is only needed for Lasso, not for Ridge

q4_answer = "___"  # Replace with A, B, C, or D

assert q4_answer != "___", "Don't forget to fill in your answer!"
assert check_answer(q4_answer, _ak['q4']), \
    "Not quite — revisit working-sessions/supervised/04_ridge_lasso_regression.ipynb. " \
    "The penalty term sums coefficients directly, so features must be on the same scale first."
print("✓ Question 4 correct!")

In [ ]:
make_grading_summary([
    (q1_answer, _ak['q1'], "Q1: How Ridge and Lasso affect coefficients differently"),
    (q2_answer, _ak['q2'], "Q2: Why Lasso performs automatic feature selection"),
    (q3_answer, _ak['q3'], "Q3: Why polynomial feature counts grow combinatorially"),
    (q4_answer, _ak['q4'], "Q4: Why regularization requires feature scaling"),
], total=4)

---
## Section B — Coding Exercises

The three exercises below compare Ridge and Lasso directly, use cross-validation to pick alpha, and combine polynomial features with regularization. Run them in order, each step builds on the previous one.

Run the setup cell first.

In [ ]:
# Shared setup — run this before the exercises
X, y = load_california_housing()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Dataset shape:", X.shape)
print("Training samples:", X_train.shape[0], " Test samples:", X_test.shape[0])

### B1 — Compare Ridge and Lasso coefficients at a fixed alpha

At the same alpha, Ridge and Lasso treat the same features very differently. Fit both and compare what each does to the coefficient vector.

In [ ]:
# B1 — Fit Ridge and Lasso at alpha=0.05 and compare coefficients

ridge = ___(alpha=0.05)     # YOUR CODE — the L2-regularized linear model
lasso = ___(alpha=0.05)     # YOUR CODE — the L1-regularized linear model

ridge.fit(___, ___)         # YOUR CODE — fit on the scaled training features and the training target
lasso.fit(___, ___)         # YOUR CODE — fit on the scaled training features and the training target

coef_compare = pd.DataFrame({
    "Feature": X.columns,
    "Ridge": ridge.coef_,
    "Lasso": lasso.coef_,
})
coef_compare["Lasso_is_zero"] = coef_compare["Lasso"] == 0

print(coef_compare.round(3).to_string(index=False))
print(f"\nLasso zeroed out {coef_compare['Lasso_is_zero'].sum()} of {len(coef_compare)} features")
print(f"Ridge zeroed out {(coef_compare['Ridge'] == 0).sum()} of {len(coef_compare)} features")

# --- checks ---
assert hasattr(ridge, 'coef_') and hasattr(lasso, 'coef_'), \
    "Both models need to be fit before inspecting coef_"
assert (coef_compare["Ridge"] != 0).all(), \
    "Ridge should not zero out any coefficients — check alpha and that you fit on scaled data"
assert coef_compare["Lasso_is_zero"].sum() >= 1, \
    "Lasso should zero out at least one coefficient at this alpha — check you fit on X_train_scaled"
print("✓ B1 complete!")

### B2 — Select alpha with cross-validation

Rather than guessing alpha, `RidgeCV` and `LassoCV` try a range of values and pick the one with the best cross-validated performance.

In [ ]:
# B2 — Use RidgeCV and LassoCV to select alpha automatically

alphas = np.logspace(-3, 3, 13)

ridge_cv = ___(alphas=___).fit(___, ___)   # YOUR CODE — the cross-validated version of the model above, given the alpha grid, fit on the scaled training data
lasso_cv = ___(alphas=___, random_state=42, max_iter=100000).fit(___, ___)  # YOUR CODE — the cross-validated version of the model above, given the same alpha grid and scaled training data

ridge_test_r2 = r2_score(y_test, ridge_cv.predict(X_test_scaled))
lasso_test_r2 = r2_score(y_test, lasso_cv.predict(X_test_scaled))

print(f"RidgeCV best alpha: {ridge_cv.alpha_:.4f}   test R²: {ridge_test_r2:.3f}")
print(f"LassoCV best alpha: {lasso_cv.alpha_:.4f}   test R²: {lasso_test_r2:.3f}")

# --- checks ---
assert ridge_cv.alpha_ in alphas, \
    "ridge_cv.alpha_ should be one of the values in the alphas grid"
assert lasso_cv.alpha_ in alphas, \
    "lasso_cv.alpha_ should be one of the values in the alphas grid"
assert ridge_test_r2 > 0.5 and lasso_test_r2 > 0.5, \
    "Test R² looks too low — check that you fit on X_train_scaled and predicted on X_test_scaled"
print("✓ B2 complete!")

### B3 — Polynomial features and the feature-count blow-up

`PolynomialFeatures` does not just add powers of each feature, it adds every interaction term too. See how quickly the feature count grows, and confirm regularization can still handle it.

In [ ]:
# B3 — Expand features with PolynomialFeatures and refit with Ridge

poly = ___(degree=___, include_bias=False)   # YOUR CODE — the feature-expansion transformer, expanded to the degree named in the markdown above
X_train_poly = poly.___(X_train)             # YOUR CODE — the method that learns the expansion and applies it in one step
X_test_poly = poly.___(X_test)               # YOUR CODE — the method that applies an already-learned expansion, without refitting

poly_scaler = StandardScaler().fit(X_train_poly)
X_train_poly_scaled = poly_scaler.transform(X_train_poly)
X_test_poly_scaled = poly_scaler.transform(X_test_poly)

ridge_poly = Ridge(alpha=1.0).fit(___, ___)   # YOUR CODE — fit on the scaled, polynomial-expanded training features and the training target
poly_test_r2 = r2_score(y_test, ridge_poly.predict(X_test_poly_scaled))

print(f"Original feature count: {X_train.shape[1]}")
print(f"Degree-2 polynomial feature count: {X_train_poly.shape[1]}")
print(f"Ridge test R² on expanded features: {poly_test_r2:.3f}")

# --- checks ---
assert X_train_poly.shape[1] == 44, \
    "Degree-2 PolynomialFeatures on 8 features (no bias) should produce 44 features"
assert poly_test_r2 > 0.5, \
    "Test R² looks too low — check the pipeline: poly -> scale -> fit -> predict"
print("✓ B3 complete!")

---
## Section C — Applied Problem

Combine everything: expand to degree-2 polynomial features, scale, and let `LassoCV` pick both the regularization strength and which of the expanded features actually matter. Fill in the blanks to run the full pipeline end to end.

In [ ]:
# Section C — Polynomial features + Lasso, end to end

# --- Step 1: Load and split ---
X_c, y_c = load_california_housing()
X_c_train, X_c_test, y_c_train, y_c_test = train_test_split(
    ___, ___, test_size=___, random_state=___   # YOUR CODE — split the features and target, holding out 20% with a fixed seed
)

# --- Step 2: Expand to degree-2 polynomial features ---
poly_c = PolynomialFeatures(degree=___, include_bias=False)   # YOUR CODE — the same degree used in B3
X_c_train_poly = poly_c.___(X_c_train)                        # YOUR CODE — the method that learns the expansion and applies it in one step
X_c_test_poly = poly_c.___(X_c_test)                           # YOUR CODE — the method that applies an already-learned expansion, without refitting

# --- Step 3: Scale (required before regularization) ---
scaler_c = StandardScaler().fit(X_c_train_poly)
X_c_train_scaled = scaler_c.transform(X_c_train_poly)
X_c_test_scaled = scaler_c.transform(X_c_test_poly)

# --- Step 4: Fit LassoCV to pick alpha and select features automatically ---
alphas_c = np.logspace(-3, 3, 13)
lasso_final = ___(alphas=___, random_state=42, max_iter=100000).fit(___, ___)  # YOUR CODE — the cross-validated Lasso, given this section's alpha grid and the scaled polynomial training features

# --- Step 5: Evaluate ---
y_c_pred = lasso_final.predict(X_c_test_scaled)
c_r2 = r2_score(___, ___)   # YOUR CODE — compare the true test target to the predictions computed above

nonzero_mask = lasso_final.coef_ != 0
n_total = len(lasso_final.coef_)
n_nonzero = nonzero_mask.sum()

print(f"Original features: {X_c_train.shape[1]}")
print(f"Degree-2 polynomial features: {n_total}")
print(f"Selected alpha: {lasso_final.alpha_:.4f}")
print(f"Features kept by Lasso: {n_nonzero} of {n_total} ({n_total - n_nonzero} eliminated)")
print(f"Test R²: {c_r2:.3f}")

# --- Step 6: Plot the top 15 non-zero coefficients by magnitude ---
feature_names_poly = poly_c.get_feature_names_out(X_c_train.columns)
kept_names = np.array(feature_names_poly)[nonzero_mask]
kept_coefs = pd.Series(lasso_final.coef_[nonzero_mask], index=kept_names)
top_coefs = kept_coefs.abs().sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(8, 6))
sns.barplot(x=top_coefs.values, y=top_coefs.index, ax=ax)
ax.set_xlabel("Absolute coefficient magnitude")
ax.set_ylabel("Feature")
ax.set_title("Top 15 Non-Zero Lasso Coefficients (Degree-2 Polynomial Features)")
plt.tight_layout()
plt.show()

# --- Section C checks ---
assert n_total == 44, \
    "Degree-2 PolynomialFeatures on 8 features (no bias) should produce 44 features"
assert n_nonzero < n_total, \
    "Lasso should eliminate at least some features — check that alpha wasn't fixed at 0"
assert c_r2 > 0.5, \
    f"R² should be above 0.5 — got {c_r2:.3f}. Check the full pipeline: poly -> scale -> fit -> predict"
print("✓ Section C complete!")
print(f"  Lasso reduced {n_total} polynomial features down to {n_nonzero}, while still explaining "
      f"{c_r2 * 100:.1f}% of the variance in house prices.")

---

## Section D — Reflection

These questions are for reflection. Edit the markdown cells below each question to write your response. There are no wrong answers, we are looking for thoughtful engagement with what you have learned. Your instructor may review these.

**Question D1**

B1 showed Ridge keeping every coefficient non-zero while Lasso zeroed several out at the same alpha. In what situation would you actually prefer Ridge's behavior over Lasso's, even though Lasso gives you automatic feature selection?

*Your response here...*

**Question D2**

Section C combined polynomial features with Lasso and still explained a similar share of variance as the plain 8-feature model, despite starting from 44 features. What does this tell you about whether the extra polynomial terms were actually adding useful information here?

*Your response here...*

**Question D3**

If you skipped the `StandardScaler` step in Section C and fit Lasso directly on the raw polynomial features, what would you expect to happen to which features get zeroed out? Why?

*Your response here...*